# ML Pipeline Frontend

Interactive interface for running ML experiments on bioinformatics datasets.
Connects to the orchestration service API at http://localhost:8000

In [1]:
import requests
import json
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

ORCHESTRATION_URL = "http://localhost:8000"
API_ENDPOINT = f"{ORCHESTRATION_URL}/api/pipeline/run"

print(f"OK Configured to use: {API_ENDPOINT}")

OK Configured to use: http://localhost:8000/api/pipeline/run


## Configure Pipeline Parameters

In [2]:
config = {
    "osd_ids": "104,48",
    "target_column": "Factor Value[Spaceflight]",
    "algorithm": "random_forest",
    "test_size": 0.2,
    "min_features": 100,
}

print("Pipeline Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

Pipeline Configuration:
  osd_ids: 104,48
  target_column: Factor Value[Spaceflight]
  algorithm: random_forest
  test_size: 0.2
  min_features: 100


## Run Pipeline

In [3]:
def run_pipeline(config):
    """Send pipeline request to orchestration service"""
    
    payload = {
        "osd_ids": config["osd_ids"],
        "config": {
            "target_column": config["target_column"],
            "task_type": "classification",
            "feature_columns": [],
            "transformations": [],
            "algorithm": config["algorithm"],
            "hyperparameters": {},
            "metrics": ["accuracy", "precision", "recall", "f1_score"],
            "test_size": config["test_size"],
            "random_state": 42,
            "min_features": config["min_features"],
            "factor_name": "Factor Value[Spaceflight]",
            "factor_values": ["Ground Control", "Space Flight"],
        }
    }
    
    print(f"Sending request to {API_ENDPOINT}")
    print(f"  OSD IDs: {config['osd_ids']}")
    print(f"  Algorithm: {config['algorithm']}")
    print(f"\nThis may take several minutes...\n")
    
    try:
        response = requests.post(
            API_ENDPOINT,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=600,
            stream=True  # ← ADD THIS for streaming
        )
        
        if response.status_code == 200:
            # Parse streaming JSON responses
            result = None
            for line in response.iter_lines():
                if line:
                    try:
                        msg = json.loads(line)
                        if msg.get('status') == 'failed':
                            print(f"\nError: {msg.get('message')}")
                            return None
                        result = msg
                    except json.JSONDecodeError:
                        pass
            
            if result:
                print("\n✓ Pipeline completed!")
                return result
            else:
                print("\nNo results received")
                return None
        else:
            print(f"\n✗ Error: {response.status_code}")
            print(response.text)
            return None
    
    except Exception as e:
        print(f"\n✗ Exception: {e}")
        return None
        
result = run_pipeline(config)


Sending request to http://localhost:8000/api/pipeline/run
  OSD IDs: 104,48
  Algorithm: random_forest

This may take several minutes...


✓ Pipeline completed!


## Display Results

In [4]:
if result:
    print("\n" + "="*70)
    print("PIPELINE RESULTS")
    print("="*70)
    
    cfg = result.get('config', {})
    print(f"\nConfiguration:")
    print(f"  Algorithm: {cfg.get('algorithm')}")
    print(f"  Target: {cfg.get('target_column')}")
    print(f"  OSD IDs: {cfg.get('osd_ids')}")
    
    train = result.get('training_results', {})
    print(f"\nTraining Results:")
    print(f"  Model ID: {train.get('model_id')}")
    print(f"  Samples: {train.get('n_samples')}")
    print(f"  Features: {train.get('n_features')}")
    
    metrics = train.get('metrics', {})
    if metrics:
        print(f"\n  Train/Test Metrics:")
        for key, value in metrics.items():
            print(f"    {key}: {value:.4f}")
    
    loocv = train.get('loocv_metrics', {})
    if loocv:
        print(f"\n  LOOCV Metrics:")
        for key, value in loocv.items():
            if key != 'n_folds':
                print(f"    {key}: {value:.4f}")
            else:
                print(f"    {key}: {value}")
else:
    print("No results")


PIPELINE RESULTS

Configuration:
  Algorithm: None
  Target: None
  OSD IDs: None

Training Results:
  Model ID: None
  Samples: None
  Features: None


## Visualize Metrics

In [5]:
if result:
    train = result.get('training_results', {})
    metrics = train.get('metrics', {})
    loocv = train.get('loocv_metrics', {})
    
    if metrics or loocv:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        if metrics:
            keys = list(metrics.keys())
            values = [metrics[k] for k in keys]
            axes[0].bar(keys, values, color='steelblue', alpha=0.8)
            axes[0].set_title('Train/Test Metrics', fontsize=14)
            axes[0].set_ylabel('Score')
            axes[0].set_ylim([0, 1])
            axes[0].grid(axis='y', alpha=0.3)
        
        if loocv:
            loocv_filtered = {k: v for k, v in loocv.items() if k != 'n_folds' and isinstance(v, float)}
            if loocv_filtered:
                keys = list(loocv_filtered.keys())
                values = [loocv_filtered[k] for k in keys]
                axes[1].bar(keys, values, color='coral', alpha=0.8)
                axes[1].set_title('LOOCV Metrics', fontsize=14)
                axes[1].set_ylabel('Score')
                axes[1].set_ylim([0, 1])
                axes[1].grid(axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
else:
    print("Run pipeline first")

## Save Results

In [6]:
if result:
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"pipeline_results_{timestamp}.json"
    
    with open(filename, 'w') as f:
        json.dump(result, f, indent=2, default=str)
    
    print(f"OK Results saved to {filename}")
    
    train = result.get('training_results', {})
    metrics = train.get('metrics', {})
    loocv = train.get('loocv_metrics', {})
    
    if metrics or loocv:
        all_metrics = {**metrics, **loocv}
        df = pd.DataFrame([all_metrics])
        csv_file = f"pipeline_metrics_{timestamp}.csv"
        df.to_csv(csv_file, index=False)
        print(f"OK Metrics saved to {csv_file}")
else:
    print("Run pipeline first")

OK Results saved to pipeline_results_20260927_212530.json
